In [ ]:
import pandas as pd
import re
import random
import os
from PIL import Image, ImageDraw, ImageFont
import cv2
import sys, os
import json
sys.path.append('../utils')
import text_utils
import image_utils
import numpy as np

salish_words, english_words = text_utils.load_wordlist("../sources/roots.csv", "../sources/english.txt")
def make_text_sample(salish_words, english_words):
    salish_sample = " ".join(random.sample(salish_words, k=random.randint(3,5)))
    english_sample = " ".join(random.sample(english_words, k=random.randint(3,5)))
    layouts = [
        salish_sample + random.choice(["."]),
        english_sample + random.choice(["."]),
        salish_sample + random.choice([".", "?"])  + "\n" + english_sample + random.choice([".", "?", "!", "…"])
    ]
    return random.choice(layouts)
os.makedirs("../new_synthetic/images", exist_ok=True)
font_cfg = json.load(open("../sources/fonts_config.json"))
all_fonts = font_cfg["Charis"] +font_cfg["Doulos"] +  font_cfg["NotoSans"]
def random_font():
    
    fpath = random.choice(all_fonts)
    size = random.randint(24, 64)
    return ImageFont.truetype(fpath, size=size)

def random_layout(img_w, img_h):
    """return margin and line spacing pattern"""
    margin = random.randint(20, 100)
    spacing = random.randint(10, 40)
    return margin, spacing

def render_text_block(text, font, img_w, img_h):
    img = Image.new("RGB", (img_w, img_h), color="white")
    draw = ImageDraw.Draw(img)
    margin, spacing = random_layout(img_w, img_h)
    y = margin
    for line in text.split("\n"):
        draw.text((margin, y), line, font=font, fill="black")
        bbox = draw.textbbox((0, 0), line, font=font)
        line_height = bbox[3] - bbox[1]
        y += line_height + spacing

    return img
entries = []
for i in range(1000):
    text = make_text_sample(salish_words, english_words)
    font = random_font()
    img = render_text_block(text, font, img_w=1400, img_h=400)
    img = image_utils.augment_image(img)
    fpath = f"../new_synthetic/images/sample_{i}.png"
    img.save(fpath)
    entries.append({'image': Image.open(fpath).convert("RGB") , "text": text})
df = pd.DataFrame(entries)
df.head()

,image,text
0,<PIL.Image.Image image mode=RGB size=1400x400 ...,'šemen'' 'sisiyus' 'acanq'émn' 'u ččí·p' 'ʔasa...
1,<PIL.Image.Image image mode=RGB size=1400x400 ...,tuition ghana dispute spam.
2,<PIL.Image.Image image mode=RGB size=1400x400 ...,se montana champion.
3,<PIL.Image.Image image mode=RGB size=1400x400 ...,raw boats research madonna global.
4,<PIL.Image.Image image mode=RGB size=1400x400 ...,investigator doctrine evaluate.


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2)
# we reset the indices to start from zero
train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

In [2]:
ipa_extras = "äáä́éɛɛ́íιóúəɔụʕʔx̣šǰčɬ∤ɫʀᴇc̕l̕m̕n̕p̕q̕r̕ṛʀ̕t̕w̕y̕wertyuiopkjhgfdsazxcvbnmʷ"

extra_tokens = list(ipa_extras)
extra_tokens

['ä',
 'á',
 'ä',
 '́',
 'é',
 'ɛ',
 'ɛ',
 '́',
 'í',
 'ι',
 'ó',
 'ú',
 'ə',
 'ɔ',
 'u',
 '̣',
 'ʕ',
 'ʔ',
 'x',
 '̣',
 'š',
 'ǰ',
 'č',
 'ɬ',
 '∤',
 'ɫ',
 'ʀ',
 'ᴇ',
 'c',
 '̕',
 'l',
 '̕',
 'm',
 '̕',
 'n',
 '̕',
 'p',
 '̕',
 'q',
 '̕',
 'r',
 '̕',
 'r',
 '̣',
 'ʀ',
 '̕',
 't',
 '̕',
 'w',
 '̕',
 'y',
 '̕',
 'w',
 'e',
 'r',
 't',
 'y',
 'u',
 'i',
 'o',
 'p',
 'k',
 'j',
 'h',
 'g',
 'f',
 'd',
 's',
 'a',
 'z',
 'x',
 'c',
 'v',
 'b',
 'n',
 'm',
 'ʷ']

In [3]:
from transformers import AutoTokenizer, VisionEncoderDecoderModel, TrOCRProcessor

tokenizer = AutoTokenizer.from_pretrained("microsoft/trocr-base-printed")
tokenizer.add_tokens(extra_tokens)

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")
model.decoder.resize_token_embeddings(len(tokenizer))


/Users/sbg/colrc-ocr-model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has o

TrOCRScaledWordEmbedding(50282, 1024, padding_idx=1)

In [ ]:
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
import os
from PIL import Image
from datasets import Dataset
dataset = Dataset.from_list(entries)

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

def preprocess(example):
    pixel_values = processor.feature_extractor(example["image"], return_tensors="pt").pixel_values[0]
    labels = tokenizer(example["text"], truncation=True, padding="max_length", max_length=128).input_ids
    example["pixel_values"] = pixel_values
    example["labels"] = labels
    return example

dataset = dataset.map(preprocess)

train_test = dataset.train_test_split(test_size=0.2)
train_dataset = train_test["train"]
val_dataset = train_test["test"]

training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr-ipa-trained",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    evaluation_strategy="steps",
    save_steps=500,
    logging_steps=100,
    num_train_epochs=10,
    fp16=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor.feature_extractor,
)

trainer.train()
